# Tutorial 03 — Analyze Visium Fluorescence Data

**Source:** https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_fluo.html  
**Dataset:** Mouse Brain coronal crop — 10x Visium Fluorescence (pre-processed, auto-download)

---

## What this notebook covers
1. Load Visium fluorescence AnnData + ImageContainer
2. Image segmentation (thresholding + watershed)
3. Extract segmentation features per spot
4. Extract and combine multi-scale summary features
5. Cluster spots based on image features
6. Compare image-based vs gene expression-based clusters

## 0. Imports

In [ ]:
%matplotlib inline

import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import squidpy as sq
import warnings
warnings.filterwarnings('ignore')

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

## 1. Load Data

In [ ]:
# Auto-downloads ~370 MB on first run
img   = sq.datasets.visium_fluo_image_crop()
adata = sq.datasets.visium_fluo_adata_crop()

print(adata)
print(f"\nClusters: {adata.obs['cluster'].nunique()}")

In [ ]:
# Visualize clusters in spatial context
sq.pl.spatial_scatter(
    adata,
    color='cluster',
    save='../results/figures/03_visium_fluo/spatial_clusters.png'
)

## 2. Image Segmentation

Squidpy's `sq.im.segment()` applies classical image segmentation to the fluorescence image. The pipeline:
1. Smooth the image with a Gaussian filter
2. Threshold to create a binary mask
3. Apply watershed to separate touching cells

In [ ]:
sq.im.process(
    img,
    layer='image',
    method='smooth',
)

sq.im.segment(
    img=img,
    layer='image_smooth',
    method='watershed',
    channel=0,
)

print("Segmentation complete.")
print(img)

In [ ]:
# Visualize segmentation result
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sq.pl.spatial_segment(
    img,
    adata,
    seg_layer='segmented_watershed',
    cell_as_circles=False,
    ax=axes[0],
)
axes[0].set_title('Segmentation result')

plt.tight_layout()
plt.savefig('../results/figures/03_visium_fluo/segmentation.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Segmentation Features

Extract per-spot features from the segmentation mask:
- **cell_count:** number of cells per spot
- **mean_intensity:** mean fluorescence per cell
- **area:** mean cell area per spot

In [ ]:
sq.im.calculate_image_features(
    adata,
    img,
    layer='image',
    key_added='features_segmentation',
    n_jobs=4,
    features='segmentation',
    features_kwargs={
        'segmentation': {
            'label_layer': 'segmented_watershed',
            'props': ['label', 'area', 'mean_intensity'],
            'channels': [0, 1],
        }
    },
)

print(f"Segmentation features shape: {adata.obsm['features_segmentation'].shape}")
print(adata.obsm['features_segmentation'].head())

In [ ]:
# Visualize cell count per spot
adata.obs['cell_count'] = adata.obsm['features_segmentation']['segmentation_label']

sq.pl.spatial_scatter(
    adata,
    color=['cluster', 'cell_count'],
    save='../results/figures/03_visium_fluo/cellcount_vs_cluster.png'
)

## 4. Multi-scale Summary Features

In [ ]:
for scale in [1.0, 2.0]:
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features='summary',
        key_added=f'features_summary_scale{scale}',
        n_jobs=4,
        scale=scale,
    )

adata.obsm['features'] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if 'features_summary' in f],
    axis='columns',
)
adata.obsm['features'].columns = ad.utils.make_index_unique(
    adata.obsm['features'].columns
)

print(f"Combined features shape: {adata.obsm['features'].shape}")

## 5. Cluster Spots by Image Features

In [ ]:
def cluster_features(features: pd.DataFrame) -> pd.Series:
    tmp = ad.AnnData(features)
    sc.pp.scale(tmp)
    sc.pp.pca(tmp, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp)
    sc.tl.leiden(tmp, key_added='leiden_img')
    return tmp.obs['leiden_img']

adata.obs['image_cluster'] = cluster_features(adata.obsm['features'])

sq.pl.spatial_scatter(
    adata,
    color=['cluster', 'image_cluster'],
    save='../results/figures/03_visium_fluo/image_vs_gene_clusters.png'
)

## 6. Save Results

In [ ]:
adata.write('../results/visium_fluo_analyzed.h5ad')
print('Saved: ../results/visium_fluo_analyzed.h5ad')